# 49. 箱线图（boxplot）

<!-- module-learning-arc:start -->
> **Seaborn 模块主线｜第 6 / 20 步：比较类别频数、水平与组内分布**
>
> **持续应用背景：** 开展客群消费行为差异研究：先固定样本和统计语义，再比较分布、关系和分面结果，判断差异是否稳定。
>
> **承接上一阶段：** 点图（pointplot）  →  **本章任务：** 箱线图（boxplot）  →  **下一步：** 小提琴图（violinplot）
>
> **大作业连接：** 本章练习将成为《客群消费行为差异研究》的一部分，最终需要从样本口径和分布比较走到关系验证、分面研究与因果边界说明。
<!-- module-learning-arc:end -->


## 本章场景

拿到一份销售或运营数据，大家习惯先看平均值，但平均值很容易被少数极端值拉偏。



## 本章目标

学完本章，你将能够：

- **理解**：理解「箱线图（boxplot）」的适用场景、数据结构要求，以及它想帮你读出的规律。
- **操作**：能按参数用相应绘图接口画出「箱线图（boxplot）」，并做必要的美化、注释与导出。
- **迁移**：能换一份真实经营数据，独立画出同类型的「箱线图（boxplot）」并读出其中的结论。


## 49.1 适用场景

**背景引入**：拿到一份销售或运营数据，大家习惯先看平均值，但平均值很容易被少数极端值拉偏。箱线图把每个分组的中位数、四分位跨度、须和离群点一次性画在图上，既能比较多组数据的“典型水平”，又能一眼看出哪组更分散、有没有异常。学会它，之后很多分组对比分析都能直接套用。

打个比方：boxplot 像'给一组数据画一张五数概括的小卡片'——箱子里的横线是中位数（中间那户人家），箱子的上下边是四分位（中间那一半人的收支范围），往两边伸的须是'还算正常'的范围，而飞出去的那些孤点，就是明显扎眼的离群值。平均值容易被人均一串极端值带偏，这张卡片却把'大多数在哪、谁在捣乱'一五一十摊开。

分类变量下比较连续数值分布。


## 49.2 数据结构

长表中的一列分类变量和一列连续数值。


## 49.3 本章练习任务

运行基础图表后，完成以下任务：

1. 添加 showfliers=False 参数，对比显示与隐藏异常点的视觉效果
2. 修改 whis 参数从默认 1.5 为 3.0，观察须范围变化对异常点数量的影响
3. 移除 hue="category" 参数，对比分组与不分组箱线图的信息密度


## 49.4 图表与参数速查

先用这张表建立本章的方法地图；每一行后面都有对应的独立示例或练习。

| 类别 | 常用方法或写法 | 主要用途 | 需要特别注意 |
| --- | --- | --- | --- |
| 基础图表 | `plt.subplots()`、`sns.boxplot()`、`ax.set()`、`fig.tight_layout()` | 分类变量下比较连续数值分布。 | 只看中位数忽略离散程度 |
| 进阶变体 | `plt.subplots()`、`sns.boxplot()`、`ax.set()`、`ax.legend()` | 在基础图表上增加分组、注释、布局或交互 | 样本很少仍隐藏原始点 |
| 关键参数 | `whis` | 须 | 只看中位数忽略离散程度 |
| 关键参数 | `hue` | 子分组 | 样本很少仍隐藏原始点 |
| 关键参数 | `order` | 类别顺序 | 须外点直接视为错误 |
| 关键参数 | `showfliers` | 异常点 | 只看中位数忽略离散程度 |


## 49.5 准备可复现数据

先完成导入和数据准备，后续单元格只负责一种图表或一种分析动作。


<!-- math-foundation:chapter-49 -->
### 数学推导｜箱线图的四分位数与异常界限

> 阅读方法：先跟着步骤理解每个量怎样产生，再看最后的可计算形式；不需要脱离业务场景死记公式。

**第 1 步｜用分位数切分排序数据。** $Q_1$、$Q_2$、$Q_3$ 分别对应累计比例 25%、50%、75%。

**第 2 步｜中间一半数据的跨度是**

$$
IQR=Q_3-Q_1
$$

**第 3 步｜把箱体向两侧延伸 1.5 个 IQR。** 下、上界分别为 $L=Q_1-1.5IQR$、$U=Q_3+1.5IQR$。箱线图的“须”通常落到界内最远的实际观测，而不是直接画到 $L$、$U$。

**把上面的关系收束为本章计算式：**

$$
IQR=Q_3-Q_1,\qquad [L,U]=[Q_1-1.5IQR,\ Q_3+1.5IQR]
$$

**符号解释：** $Q_1$、$Q_3$ 是第一和第三四分位数，IQR 描述中间 50% 数据的跨度。

**代码对应：** 用 `quantile([.25, .5, .75])` 复核图中的箱体和中位数。

**使用边界：** 落在界限外的是统计异常点，不等于错误数据，更不能自动删除。


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
import numpy as np

# 中文字体支持：由平台运行时自动配置
# 说明：Matplotlib 默认字体不含中文字形，中文会显示成方框。
#      本平台在运行每个绘图 cell 前，会自动注册可用的中文字体并设置
#      font.sans-serif / axes.unicode_minus，即使 seaborn 的 sns.set_theme
#      会重置字体，运行时也会在 set_theme 之后自动恢复。因此这里无需手动
#      import 或 addfont，直接使用即可。

# 1️⃣ 主题与数据导入：统一画风，读取三个公开数据集
sns.set_theme(style="whitegrid", context="notebook")

diamonds = pd.read_csv("/datasets/diamonds.csv")
taxis = pd.read_csv("/datasets/taxis.csv", parse_dates=["pickup", "dropoff"])
flights = pd.read_csv("/datasets/flights.csv")
print(f"Diamonds {len(diamonds):,} | Taxis {len(taxis):,} | Flights {len(flights):,} 行")


In [ ]:
# 2️⃣ 特征工程：把原始字段映射成图表统一使用的列名与派生指标
orders_full = diamonds.assign(
    category=diamonds["cut"],
    channel=diamonds["color"],
    region=diamonds["clarity"],
    order_value=diamonds["price"],
    items=diamonds["carat"],
    satisfied=np.where(
        diamonds["price"] >= diamonds["price"].median(), "高于中位价", "不高于中位价"
    ),
)
orders = orders_full.sample(2_000, random_state=36)

marketing_full = taxis.assign(
    channel=taxis["payment"].fillna("unknown"),
    visits=taxis["distance"],
    ad_spend=taxis["tip"],
    sales=taxis["total"],
    conversion=(taxis["tip"] / taxis["total"].replace(0, np.nan)).fillna(0),
)
marketing = marketing_full.sample(
    min(2_000, len(marketing_full)), random_state=36
).copy()

daily = flights.assign(
    date=pd.to_datetime(
        flights["year"].astype(str) + "-" + flights["month"] + "-01"
    ),
    region="AirPassengers",
    sales=flights["passengers"],
)
print(f"样本：orders {len(orders):,} | marketing {len(marketing):,} | daily {len(daily):,} 行")


## 49.6 基础图表

先保留必要的编码：位置、颜色或大小。图表标题、坐标轴和单位应能让读者脱离代码理解结果。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(
    data=orders,
    x="category",
    y="order_value",
    hue="category",
    palette="Set2",
    legend=False,
    ax=ax,
)
ax.set(title="品类客单价分布", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()


**练一练**：基础图表就用默认的 `whis=1.5` 画出了每类客单价的“须”。试着把须倍数改成 3.0，箱子的须范围会变宽，原本落在须外的异常点会变少。请补全代码，把 `whis` 改成 3.0 重新绘制箱线图，并用程序统计新须范围外还剩多少个异常点。


In [ ]:
# 请在下方填写代码
import matplotlib.pyplot as plt
import seaborn as sns

# 任务：对照 43.4 基础图表，把默认的须倍数 whis=1.5 改成 3.0，
# 重新绘制箱线图，并观察须范围变大后异常点数量如何变化。

# 第一步：把下面的变量设为 3.0
whis_x = None  # ← 请填 3.0

# 第二步：按提示补全绘图代码（在 sns.boxplot 中加入 whis=whis_x）
fig, ax = plt.subplots(figsize=(8, 4.5))
if whis_x is not None:
    sns.boxplot(
        data=orders,
        x="category",
        y="order_value",
        hue="category",
        palette="Set2",
        legend=False,
        whis=whis_x,
        ax=ax,
    )
    ax.set(title="须倍数为 3.0 的客单价箱线图", xlabel="品类", ylabel="客单价（元）")
plt.show()

# 第三步：程序统计 whis=3.0 时落在须范围之外的异常点数量
if whis_x is not None:
    q1 = orders.groupby("category")["order_value"].quantile(0.25)
    q3 = orders.groupby("category")["order_value"].quantile(0.75)
    lower = orders["category"].map(q1 - whis_x * (q3 - q1))
    upper = orders["category"].map(q3 + whis_x * (q3 - q1))
    n_outliers = int((~(orders["order_value"].between(lower, upper))).sum())
else:
    n_outliers = None


In [ ]:
# ===== 参考答案 =====
import matplotlib.pyplot as plt
import seaborn as sns

# 思路：只把默认的 whis 由 1.5 改为 3.0，其余编码保持不变，
# 再统计须范围扩大后仍落在箱外（异常）的观测数量。
whis_x = 3.0

fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(
    data=orders,
    x="category",
    y="order_value",
    hue="category",
    palette="Set2",
    legend=False,
    whis=whis_x,
    ax=ax,
)
ax.set(title="须倍数为 3.0 的客单价箱线图", xlabel="品类", ylabel="客单价（元）")
plt.show()

# 程序计算落在新须范围之外的异常点数量
q1 = orders.groupby("category")["order_value"].quantile(0.25)
q3 = orders.groupby("category")["order_value"].quantile(0.75)
lower = orders["category"].map(q1 - whis_x * (q3 - q1))
upper = orders["category"].map(q3 + whis_x * (q3 - q1))
n_outliers = int((~(orders["order_value"].between(lower, upper))).sum())


## 49.7 进阶变体

在基础图表可读的前提下增加分组、布局、注释或交互。新增编码必须服务于一个明确问题。


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(9, 4.8))
sns.boxplot(
    data=orders,
    x="region",
    y="order_value",
    hue="channel",
    palette="colorblind",
    ax=ax,
)
ax.set(title="区域与渠道客单价分布", xlabel="区域", ylabel="客单价（元）")
ax.legend(title="渠道", frameon=False, ncol=3)
fig.tight_layout()
plt.show()


## 49.8 参数说明

- whis：须
- hue：子分组
- order：类别顺序
- showfliers：异常点


## 49.9 结果解读

比较中位数、箱体宽度和须外点；结合原始散点判断样本密度。


## 49.10 本章实训：分组比较与不确定性

这一组实验专门训练“观察一个结果 → 只改一个变量 → 解释变化”。先运行第一个代码单元格，再运行第二个。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame(
    {
        "region": ["华东", "华南", "华北", "西南"],
        "sales": [320, 250, 280, 190],
    }
)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="region", y="sales", ci=None, ax=ax, color="#0F766E"
)
ax.set_title("地区销售额比较")
ax.set_ylabel("销售额")
plt.show()


### 49.10.1 第一个结果怎么读

Seaborn 负责把 DataFrame 的字段映射为图形编码；先明确横轴、纵轴和每行数据的粒度，再选择图表。

请记录：输入是什么、输出是什么、输出支持了哪一个结论。


In [ ]:
report = report.sort_values("sales", ascending=False)
fig, ax = plt.subplots(figsize=(7, 3.5))
sns.barplot(
    data=report, x="sales", y="region", ci=None, ax=ax, color="#F59E0B"
)
ax.set_title("按销售额排序的地区比较")
ax.set_xlabel("销售额")
ax.set_ylabel("地区")
plt.show()


### 49.10.2 第二个结果怎么读

第二个实验只改变排序和坐标方向，让读者更容易找到最大值。图表调整必须服务于阅读任务。

迁移任务：把一个输入值、一个字段或一个图表参数换成自己的例子，再用一句话解释变化。


## 49.11 错误恢复：分组字段缺失怎么办

真实数据和真实代码都会出问题。本节先观察问题，再用一个明确的检查或修复步骤恢复运行。


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

report = pd.DataFrame({"region": ["华东", "华南"], "sales": [120, 150]})
required = {"region", "sales"}
missing = required - set(report.columns)
if missing:
    print("缺少字段：", sorted(missing))
else:
    fig, ax = plt.subplots(figsize=(6, 3))
    sns.barplot(data=report, x="region", y="sales", ci=None, ax=ax)
    ax.set_title("地区销售额")
    plt.show()


### 49.11.1 错误恢复步骤

1. 先看错误类型、字段或数据形状。
2. 判断问题发生在输入、处理中间结果还是输出。
3. 修复后重新检查结果，而不是只让代码不报错。

绘图前先检查字段是否存在。把字段检查放在画图之前，错误会更接近真正原因，也更容易恢复。

迁移任务：把示例中的输入换成一组会触发问题的数据，并记录你的修复规则。


## 49.12 易错点提醒

- 只看中位数忽略离散程度
- 样本很少仍隐藏原始点
- 须外点直接视为错误


## 49.13 练习与作业

请使用同一份数据完成下面任务，并说明你选择该图表的原因。完成后补充：图表回答了什么问题、最重要的视觉信号是什么、还有哪些信息无法从图中得出。


## 49.14 独立迁移练习

修改一个分组、排序或统计设置，并比较修改前后的结论。

先在下面单元格完成自己的版本；需要参考时再回看紧邻的示例或参考实现。


In [ ]:
# 独立迁移练习：把箱线图与抖动散点叠加，同时看分布与原始点
# 【目标】箱线看摘要、抖动看原始点，二者叠加信息互补。
import matplotlib.pyplot as plt
import seaborn as sns

# 起点示例(已可运行)：先画箱线(隐藏离群点)，再叠加 stripplot 显示原始观察。
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.boxplot(
    data=orders, x="category", y="order_value", color="#e8eaed", showfliers=False, ax=ax
)
sns.stripplot(
    data=orders.sample(300, random_state=7),
    x="category",
    y="order_value",
    color="#1a73e8",
    size=3,
    alpha=0.6,
    ax=ax,
)
ax.set(title="品类客单价分布与原始点", xlabel="品类", ylabel="客单价（元）")
fig.tight_layout()
plt.show()

# ---- 反思记录：叠加原始点后，箱线摘要之外还看到了什么 ----
change_note = "待填写"
expected_change = "待填写"
observed_change = "运行后填写"
print(f"改动：{change_note}")
print(f"预期：{expected_change}")
print(f"观察：{observed_change}")


In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(8.5, 4.5))
sns.boxplot(
    data=orders,
    x="satisfied",
    y="order_value",
    hue="category",
    palette="Set2",
    ax=ax,
)
ax.set(title="评价与客单价分布", xlabel="评价", ylabel="客单价（元）")
ax.legend(title="品类", frameon=False)
fig.tight_layout()
plt.show()


## 49.15 小结

用Seaborn箱线图比较分类组的中位数、四分位距和潜在异常。


### 49.15.1 你已经掌握

- 判断箱线图（boxplot）的适用场景
- 准备与图表匹配的数据结构
- 从基础图表扩展到分组、注释或交互变体
- 按照业务问题解读图表并说明结论边界


### 49.15.2 关键参数

| 参数 | 作用 |
| --- | --- |
| `whis` | 须 |
| `hue` | 子分组 |
| `order` | 类别顺序 |
| `showfliers` | 异常点 |


### 49.15.3 需要注意

- 只看中位数忽略离散程度
- 样本很少仍隐藏原始点
- 须外点直接视为错误


### 49.15.4 完成检查

- [ ] 能判断什么问题适合使用箱线图（boxplot）
- [ ] 能准备符合要求的数据结构
- [ ] 能独立完成基础图表和一个进阶变体
- [ ] 能调整关键参数并解释视觉变化
- [ ] 能根据图表写出有边界的数据结论


### 49.15.5 下一步推荐

把同一图表迁移到另一份数据，先保留同样的编码，再只改变一个维度。比较迁移前后的可读性，并说明哪些结论仍然成立。
